# 03 — Backtest MACD+RSI Confluence

Run the MACD+RSI confluence strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Strategy logic:**
- **Entry:** MACD line crosses signal line AND RSI confirms momentum direction
- **Exit:** MACD reversal crossover (can flip to opposite side) OR RSI extreme (take-profit)
- NT's RSI uses **0.0–1.0 scale** (not 0–100). RSI 70 = `0.70`.
- NT's MACD only computes `fast_MA - slow_MA`; signal line is a separate EMA of MACD values.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + MACD + RSI panels, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweeps — MACD period grid, RSI threshold grid

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.indicators import (
    ExponentialMovingAverage,
    MovingAverageConvergenceDivergence,
    RelativeStrengthIndex,
)
from nautilus_trader.model.currencies import USDC
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_single_backtest
from src.core import bar_type_str
from src.strategies.macd_rsi import MACDRSI, MACDRSIConfig

from charts import plot_macd_rsi, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html

# ── Shared config ────────────────────────────────────────────────
CATALOG_PATH     = "../data/catalog"
INSTRUMENT_ID    = "BTC-USD-PERP.HYPERLIQUID"
BAR_INTERVAL     = "1h"
BAR_TYPE_STR     = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE            = Venue("HYPERLIQUID")
STARTING_CAPITAL = 10_000
TRADE_SIZE       = Decimal("0.01")   # 0.01 BTC per trade

# MACD params
MACD_FAST   = 12
MACD_SLOW   = 26
MACD_SIGNAL = 9

# RSI params (NT RSI uses 0.0-1.0 scale!)
RSI_PERIOD          = 14
RSI_OVERBOUGHT      = 0.70
RSI_OVERSOLD        = 0.30
RSI_ENTRY_THRESHOLD = 0.50

# Parameter sweep grids
MACD_FAST_PERIODS = [8, 12, 16]
MACD_SLOW_PERIODS = [20, 26, 34]
RSI_OB_VALUES     = [0.65, 0.70, 0.75, 0.80]
RSI_OS_VALUES     = [0.20, 0.25, 0.30, 0.35]

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

print(f"Instrument : {instrument.id}")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add MACDRSI strategy + run ────────────────────────────
config = MACDRSIConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=TRADE_SIZE,
    macd_fast_period=MACD_FAST,
    macd_slow_period=MACD_SLOW,
    macd_signal_period=MACD_SIGNAL,
    rsi_period=RSI_PERIOD,
    rsi_overbought=RSI_OVERBOUGHT,
    rsi_oversold=RSI_OVERSOLD,
    rsi_entry_threshold=RSI_ENTRY_THRESHOLD,
)
strategy = MACDRSI(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"MACD({MACD_FAST}/{MACD_SLOW}/{MACD_SIGNAL}) + RSI({RSI_PERIOD}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

In [ ]:
# ── Cell 8: Plotly chart ───────────────────────────────────────────

fig = plot_macd_rsi(
    bars, fills_report,
    macd_fast=MACD_FAST,
    macd_slow=MACD_SLOW,
    macd_signal=MACD_SIGNAL,
    rsi_period=RSI_PERIOD,
    rsi_overbought=RSI_OVERBOUGHT,
    rsi_oversold=RSI_OVERSOLD,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"MACD({MACD_FAST}/{MACD_SLOW}/{MACD_SIGNAL}) + RSI({RSI_PERIOD})  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ───────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions))

In [ ]:
# ── Cell 11: Parameter sweep — MACD periods ──────────────────────
#
# Grid search over MACD fast/slow period combinations.
# RSI params are held at defaults.

def add_macd_rsi_strategy(eng, fast, slow, signal=MACD_SIGNAL,
                          rsi_period=RSI_PERIOD, rsi_ob=RSI_OVERBOUGHT,
                          rsi_os=RSI_OVERSOLD, rsi_entry=RSI_ENTRY_THRESHOLD):
    cfg = MACDRSIConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_size=TRADE_SIZE,
        macd_fast_period=fast,
        macd_slow_period=slow,
        macd_signal_period=signal,
        rsi_period=rsi_period,
        rsi_overbought=rsi_ob,
        rsi_oversold=rsi_os,
        rsi_entry_threshold=rsi_entry,
    )
    eng.add_strategy(MACDRSI(cfg))

# ── Run the MACD sweep ──
results = []
combos = [(f, s) for f in MACD_FAST_PERIODS for s in MACD_SLOW_PERIODS if f < s]
total = len(combos)

for i, (fast, slow) in enumerate(combos, 1):
    row = run_single_backtest(
        venue=VENUE, instrument=instrument, bars=bars,
        starting_capital=STARTING_CAPITAL,
        params={"fast": fast, "slow": slow},
        add_strategy=lambda eng, f=fast, s=slow: add_macd_rsi_strategy(eng, f, s),
    )
    results.append(row)
    status = f"  !! {row['error']}" if row.get("error") else ""
    print(
        f"  [{i}/{total}] fast={fast:>3}, slow={slow:>3}  "
        f"PnL={row['total_pnl']:>10.2f}  PnL%={row['total_pnl_pct']:>7.2f}%"
        f"  positions={row['num_positions']}{status}"
    )

results_df = pd.DataFrame(results)

display_cols = ["fast", "slow", "total_pnl", "total_pnl_pct",
                "num_positions", "final_balance", "min_balance", "error"]
for col in results_df.columns:
    lower = col.lower()
    if any(kw in lower for kw in ["sharpe", "drawdown", "win", "trades", "expectancy", "factor"]):
        display_cols.append(col)

display_cols = [c for c in display_cols if c in results_df.columns]
print("\n=== MACD Parameter Sweep Results ===")
display(results_df[display_cols])

In [ ]:
# ── Cell 12: PnL heatmap (MACD fast vs slow) ─────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label="MACD Slow Period", col_label="MACD Fast Period",
    title="Total PnL (USDC) by MACD Parameters",
)

In [ ]:
# ── Cell 13: RSI sensitivity sweep ────────────────────────────────
#
# Fix MACD at the best combo from the previous sweep, vary RSI thresholds.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
best_fast = int(best_row["fast"])
best_slow = int(best_row["slow"])
print(f"Best MACD params: fast={best_fast}, slow={best_slow} (PnL={best_row['total_pnl']:,.2f})")
print(f"Sweeping RSI overbought={RSI_OB_VALUES} x oversold={RSI_OS_VALUES}\n")

rsi_results = []
combos = [(ob, os_) for ob in RSI_OB_VALUES for os_ in RSI_OS_VALUES]
total = len(combos)

for i, (ob, os_) in enumerate(combos, 1):
    row = run_single_backtest(
        venue=VENUE, instrument=instrument, bars=bars,
        starting_capital=STARTING_CAPITAL,
        params={"rsi_ob": ob, "rsi_os": os_},
        add_strategy=lambda eng, _ob=ob, _os=os_: add_macd_rsi_strategy(
            eng, best_fast, best_slow, rsi_ob=_ob, rsi_os=_os,
        ),
    )
    rsi_results.append(row)
    status = f"  !! {row['error']}" if row.get("error") else ""
    print(
        f"  [{i}/{total}] OB={ob:.2f}, OS={os_:.2f}  "
        f"PnL={row['total_pnl']:>10.2f}  PnL%={row['total_pnl_pct']:>7.2f}%{status}"
    )

rsi_df = pd.DataFrame(rsi_results)

plot_pnl_heatmap(
    rsi_df, row_col="rsi_ob", col_col="rsi_os",
    row_label="RSI Overbought", col_label="RSI Oversold",
    title=f"Total PnL (USDC) by RSI Thresholds — MACD({best_fast}/{best_slow})",
)

In [ ]:
# ── Cell 14: TradingView Interactive Backtest ──────────────────────
#
# EMA overlay uses MACD's underlying fast/slow periods — shows the
# moving averages that MACD internally computes the difference of.

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=MACD_FAST,
    slow_period=MACD_SLOW,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    output_path=f"../reports/macd_rsi_{MACD_FAST}_{MACD_SLOW}_{MACD_SIGNAL}_{BAR_INTERVAL}.html",
)
# Then just open the file in your browser

In [ ]:
# ── Cell 15: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")